# Лабораторная работа №3. Решающее дерево

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются те же Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE), что в [лабораторной №1](lab1.ipynb). Тест и фолды не меняются. Основные метрики подбора: macro-F1 вверх, MAE вниз.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует исходные числовые признаки.

In [1]:
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'raw'

def sklearn_model(task, params):
    if task == 'classification':
        return DecisionTreeClassifier(random_state=SEED, **params)
    return DecisionTreeRegressor(random_state=SEED, **params)

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [2]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.8736,0.8864,0.0020,NaN,NaN
1,regression,NaN,NaN,0.0063,32.0561,84.0411


## Гипотезы и кросс-валидация

Дерево без ограничений может подогнаться под редких покупателей и отдельные замеры. Проверяю ограничения глубины и минимального размера листа. Отдельно проверяю доли расходов и составные признаки плитки: одиночные монотонные логарифмы сами по себе почти не меняют порядок порогов.

В обоих случаях 5 фолдов. Сравниваю препроцессинг без подбора, подбор без изменения признаков и их комбинацию. Сначала выбираю лучшую альтернативу бейзлайну на CV, затем оцениваю её на отложенной части. `cv_std` показывает разброс оценок по фолдам, это не доверительный интервал.

In [3]:
feature_options = ['features']
class_grid = [{'max_depth': 3, 'min_samples_leaf': 2}, {'max_depth': 3, 'min_samples_leaf': 5}, {'max_depth': 3, 'min_samples_leaf': 10}, {'max_depth': 6, 'min_samples_leaf': 2}, {'max_depth': 6, 'min_samples_leaf': 5}, {'max_depth': 6, 'min_samples_leaf': 10}, {'max_depth': 10, 'min_samples_leaf': 2}, {'max_depth': 10, 'min_samples_leaf': 5}, {'max_depth': 10, 'min_samples_leaf': 10}]
reg_grid = [{'max_depth': 3, 'min_samples_leaf': 2}, {'max_depth': 3, 'min_samples_leaf': 5}, {'max_depth': 3, 'min_samples_leaf': 10}, {'max_depth': 6, 'min_samples_leaf': 2}, {'max_depth': 6, 'min_samples_leaf': 5}, {'max_depth': 6, 'min_samples_leaf': 10}, {'max_depth': 10, 'min_samples_leaf': 2}, {'max_depth': 10, 'min_samples_leaf': 5}, {'max_depth': 10, 'min_samples_leaf': 10}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

classification


,experiment,prep,params,cv_mean,cv_std
0,parameters,raw,"{'max_depth': 3, 'min_samples_leaf': 5}",0.8958,0.0296
1,combined,features,"{'max_depth': 3, 'min_samples_leaf': 5}",0.8892,0.0240
2,preprocessing,features,{},0.8555,0.0533
3,baseline,raw,{},0.8367,0.0437


regression


,experiment,prep,params,cv_mean,cv_std
0,preprocessing,features,{},28.1049,2.0807
1,combined,features,"{'max_depth': 10, 'min_samples_leaf': 2}",29.6958,2.5783
2,baseline,raw,{},29.8516,1.8780
3,parameters,raw,"{'max_depth': 10, 'min_samples_leaf': 2}",30.0716,2.1951


Лучший вариант из альтернатив заново обучаю на всём train. Прирост на CV не гарантирует прироста на тесте.

In [4]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.9004,0.9091,0.0022,NaN,NaN
1,regression,NaN,NaN,0.0111,24.6829,57.7775


## Самописная реализация

На каждом узле перебираются границы между различными значениями признака. Критерии: уменьшение взвешенной примеси Джини и уменьшение суммы квадратов ошибок. Сортировка и накопительные суммы позволяют проверить все пороги за один проход после сортировки. Лист хранит моду класса или среднее целевой переменной.

Самописный алгоритм проверяется с обоими рецептами sklearn: исходным и выбранным по CV. Приёмы обработки признаков общие, чтобы сравнить реализацию модели, а не разные входные данные.

In [5]:
from models.tree import Tree

def own_model(task, params):
    return Tree(task, **params)

own_base = {task: evaluate(task, own_model(task, {}), BASE) for task in TASKS}
own_improved = {task: evaluate(task, own_model(task, row.params), row.prep)
                for task, row in chosen.items()}

Сохраняю четыре сравнимые строки на задачу. `Δ` ниже означает разницу с соответствующим библиотечным или исходным вариантом, а не процентное изменение. Для MAE отрицательная разница означает улучшение.

In [6]:
versions = {'sklearn baseline': baseline, 'sklearn improved': improved,
            'own baseline': own_base, 'own improved': own_improved}
scores = pd.DataFrame([dict(version=version, **metrics)
                       for version, outcomes in versions.items()
                       for metrics, _ in outcomes.values()])
from pathlib import Path

Path('results').mkdir(exist_ok=True)
scores.drop(columns='fit_seconds').to_csv('results/lab3.csv', index=False)
for task, metric in zip(TASKS, ('f1_macro', 'mae_ms')):
    subset = scores.query('task == @task').dropna(axis=1).set_index('version')
    display(subset.drop(columns='task').round(4))
    s = subset[metric]
    display(pd.Series({'sklearn improved − sklearn baseline': s.iloc[1] - s.iloc[0],
                       'own baseline − sklearn baseline': s.iloc[2] - s.iloc[0],
                       'own improved − sklearn improved': s.iloc[3] - s.iloc[1],
                       'own improved − own baseline': s.iloc[3] - s.iloc[2]},
                      name=f'Δ {metric}').round(4))

,f1_macro,accuracy,fit_seconds
version,,,
sklearn baseline,0.8736,0.8864,0.0020
sklearn improved,0.9004,0.9091,0.0022
own baseline,0.8527,0.8636,0.0070
own improved,0.9004,0.9091,0.0023


sklearn improved − sklearn baseline    0.0268
own baseline − sklearn baseline       -0.0209
own improved − sklearn improved        0.0000
own improved − own baseline            0.0477
Name: Δ f1_macro, dtype: float64

,fit_seconds,mae_ms,rmse_ms
version,,,
sklearn baseline,0.0063,32.0561,84.0411
sklearn improved,0.0111,24.6829,57.7775
own baseline,0.4602,31.1828,80.1018
own improved,0.6344,23.8860,55.9817


sklearn improved − sklearn baseline   -7.3732
own baseline − sklearn baseline       -0.8733
own improved − sklearn improved       -0.7968
own improved − own baseline           -7.2968
Name: Δ mae_ms, dtype: float64

Сравниваю ошибки обеих улучшенных моделей на тех же тестовых объектах. Матрица показывает, какие классы путаются. В регрессии сравниваю величину ошибки на быстрых и медленных конфигурациях; медианный порог беру из train.

In [7]:
from sklearn.metrics import confusion_matrix

_, _, _, test_channel = split('classification')
_, _, train_time, test_time = split('regression')
bands = np.where(test_time <= train_time.median(), 'быстрые', 'медленные')
for label, outcomes in [('sklearn', improved), ('самописная', own_improved)]:
    print(label)
    display(pd.DataFrame(confusion_matrix(test_channel, outcomes['classification'][1], labels=[1, 2]),
                         index=['факт Horeca', 'факт Retail'],
                         columns=['прогноз Horeca', 'прогноз Retail']))
    error = np.abs(np.asarray(test_time) - outcomes['regression'][1])
    display(pd.Series(error).groupby(bands).agg(['size', 'mean']).round(2))

sklearn


,прогноз Horeca,прогноз Retail
факт Horeca,53,7
факт Retail,1,27


,size,mean
быстрые,308,7.59
медленные,292,42.72


самописная


,прогноз Horeca,прогноз Retail
факт Horeca,53,7
факт Retail,1,27


,size,mean
быстрые,308,7.89
медленные,292,40.76


## Выводы

- **Классификация.** Ограничения `max_depth=3` и `min_samples_leaf=5` подняли CV macro-F1 с 0,8367 до 0,8958. Тест: 0,8736 → 0,9004 у sklearn и 0,8527 → 0,9004 у самописного дерева. Глубокие деревья могут расходиться из-за выбора между разбиениями с одинаковым уменьшением примеси; на выбранной глубине разница по метрикам исчезла.
- **Регрессия.** Выиграл препроцессинг без настройки глубины: CV MAE 28,10 вместо 29,85 мс. На тесте MAE sklearn снизилась с 32,06 до 24,68 мс, а самописного дерева с 31,18 до 23,89 мс. Этот разрыв между реализациями на одном тесте не означает, что самописный алгоритм стабильно точнее.